# Clinical cases — unified SOTA + GPT-OSS annotation notebook

Clean production notebook for the clinical cases bundle.

Goal:

1. Start from `clinical_cases_bundle.duckdb`.
2. Build the quantitative SOTA semantic layer with embedding models, neighbors, consensus, clustering, UMAP coordinates, topic terms, diversity metrics, and `star_case_scores`.
3. Run `openai/gpt-oss-20b` as a structured annotation model, not as an embedding model.
4. Save one unified bundle with both layers.

___

GPU recommended + A100

In [ ]:
# =============================
# 1. Runtime configuration
# =============================
from pathlib import Path

ARCHIVO_NOMBRE = "clinical_cases_bundle.duckdb"

# Input: clean canonical bundle produced by the OCR/curation pipeline.
BUNDLE_PATH = Path(ARCHIVO_NOMBRE)

# Output: one final enriched bundle containing BOTH:
#   A) SOTA quantitative semantic layer
#   B) GPT-OSS structured annotation layer
OUTPUT_BUNDLE_PATH = Path("clinical_cases_bundle_sota_gptoss20b.duckdb")
OUTPUT_DIR = Path("outputs_sota_gptoss20b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Execution switches
# -----------------------------
RUN_MODE = "full"                 # allowed: "smoke", "full"
RUN_SOTA = True                   # embeddings, neighbors, clusters, UMAP tables, star_case_scores
RUN_LLM_ANNOTATION = True         # GPT-OSS annotations, not embeddings
ENABLE_PLOTS = True              # keep False for a clean production run
FORCE_REWRITE_OUTPUT_BUNDLE = True

# -----------------------------
# Embedding/SOTA profile
# -----------------------------
ANALYSIS_PROFILE = "full_sota"    # allowed: "smoke", "full_sota", "max_power"
RANDOM_SEED = 42
BATCH_SIZE = 4
MAX_BATCH_SIZE = 16
USE_FP16_ON_CUDA = True
ALLOW_MODEL_FALLBACK = True
TRUST_REMOTE_CODE = True

# Full-text chunking
CHUNK_TOKENS = 768
CHUNK_OVERLAP = 256
MIN_CHUNK_CHARS = 200
SMOKE_N_CASES = 12
SMOKE_LIMIT_CHUNKS = None

# Neighbor/projection/cluster sweep
NEIGHBOR_TOP_K = 10
UMAP_NEIGHBORS_GRID = [5, 10, 15, 30, 50]
UMAP_MIN_DIST_GRID = [0.0, 0.05, 0.1, 0.3, 0.6]
PROJECTION_SEEDS = [0, 1, 2, 3, 4]
KMEANS_K_GRID = list(range(4, 13))
AGGLOMERATIVE_K_GRID = list(range(4, 13))

# Embedding models. GPT-OSS is intentionally NOT here; it is used later for annotation.
EMBEDDING_MODEL_REGISTRY = {
    "qwen3_8b": {
        "model_id": "Qwen/Qwen3-Embedding-8B",
        "backend": "transformers_mean_pool",
        "instruction": "Represent this Spanish clinical-laboratory teaching case for semantic similarity, curriculum coverage, and case-based learning retrieval: ",
        "priority": 1,
    },
    "qwen3_4b": {
        "model_id": "Qwen/Qwen3-Embedding-4B",
        "backend": "transformers_mean_pool",
        "instruction": "Represent this Spanish clinical-laboratory teaching case for semantic similarity and learning analysis: ",
        "priority": 2,
    },
    "nvidia_llama_nemotron_8b": {
        "model_id": "nvidia/Llama-Embed-Nemotron-8B",
        "backend": "sentence_transformers_or_transformers",
        "instruction": "Represent this Spanish clinical-laboratory teaching case for semantic retrieval and clustering: ",
        "priority": 3,
    },
    "nv_embed_v2": {
        "model_id": "nvidia/NV-Embed-v2",
        "backend": "sentence_transformers_or_transformers",
        "instruction": "Represent this Spanish clinical-laboratory teaching case for semantic retrieval and clustering: ",
        "priority": 4,
    },
    "bge_m3": {
        "model_id": "BAAI/bge-m3",
        "backend": "sentence_transformers",
        "instruction": "Represent this Spanish clinical-laboratory teaching case for semantic retrieval: ",
        "priority": 5,
    },
    "jina_v3": {
        "model_id": "jinaai/jina-embeddings-v3",
        "backend": "sentence_transformers",
        "instruction": "Represent this Spanish clinical-laboratory teaching case for semantic retrieval: ",
        "priority": 6,
    },
    "e5_baseline": {
        "model_id": "intfloat/multilingual-e5-base",
        "backend": "sentence_transformers",
        "instruction": "passage: ",
        "priority": 99,
    },
}

PROFILE_MODELS = {
    "smoke": ["bge_m3", "e5_baseline"],
    "full_sota": ["qwen3_8b", "nvidia_llama_nemotron_8b", "bge_m3"],
    "max_power": ["qwen3_8b", "nvidia_llama_nemotron_8b", "nv_embed_v2", "bge_m3", "jina_v3"],
}

SELECTED_EMBEDDING_KEYS = PROFILE_MODELS[ANALYSIS_PROFILE]

# -----------------------------
# GPT-OSS annotation config
# -----------------------------
LLM_MODEL_REGISTRY = {
    "gpt_oss_20b": "openai/gpt-oss-20b",
    "gpt_oss_120b": "openai/gpt-oss-120b",
    "qwen3_8b_instruct": "Qwen/Qwen3-8B",
    "qwen3_14b_instruct": "Qwen/Qwen3-14B",
}
LLM_MODEL_KEY = "gpt_oss_20b"

# GPT-OSS produces annotations, not vector embeddings.
LLM_ANNOTATION_N_CASES = 139 if RUN_MODE == "full" else min(12, SMOKE_N_CASES)
LLM_SELECTION_MODE = "star_first"  # allowed: "star_first", "book_order"
TEST_ONE_CASE_FIRST = False

LLM_MAX_INPUT_CHARS = 12000
LLM_RETRY_MAX_INPUT_CHARS = 8000
LLM_MAX_PROMPT_TOKENS = 24000
LLM_MAX_NEW_TOKENS = 2200
DO_SAMPLE = False
TEMPERATURE = None
TOP_P = None
RETRY_FAILED_JSON_ONCE = True
PROMPT_VERSION = "llm_annotation_v2_strict_json_retry"

print({
    "BUNDLE_PATH": str(BUNDLE_PATH),
    "OUTPUT_BUNDLE_PATH": str(OUTPUT_BUNDLE_PATH),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "RUN_MODE": RUN_MODE,
    "RUN_SOTA": RUN_SOTA,
    "RUN_LLM_ANNOTATION": RUN_LLM_ANNOTATION,
    "ANALYSIS_PROFILE": ANALYSIS_PROFILE,
    "SELECTED_EMBEDDING_KEYS": SELECTED_EMBEDDING_KEYS,
    "LLM_MODEL_KEY": LLM_MODEL_KEY,
    "LLM_ANNOTATION_N_CASES": LLM_ANNOTATION_N_CASES,
})

In [ ]:
# =============================
# 2. Dependency installation
# =============================
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "duckdb": "duckdb",
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "transformers": "transformers",
    "sentence_transformers": "sentence-transformers",
    "umap": "umap-learn",
    "hdbscan": "hdbscan",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "scipy": "scipy",
    "accelerate": "accelerate",
    "safetensors": "safetensors",
}

OPTIONAL_PACKAGES = {
    "json_repair": "json-repair",
    "bitsandbytes": "bitsandbytes",
    "triton": "triton",
    "kernels": "kernels",
}

def install_if_missing(import_name, pip_name):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
    else:
        print(f"OK: {pip_name}")

for import_name, pip_name in REQUIRED_PACKAGES.items():
    install_if_missing(import_name, pip_name)

print("Optional packages available check:")
for import_name, pip_name in OPTIONAL_PACKAGES.items():
    print(f"  {pip_name}: {importlib.util.find_spec(import_name) is not None}")


In [ ]:
# =============================
# 3. Imports, seed, GPU diagnostics
# =============================
import os
import gc
import json
import math
import time
import shutil
import hashlib
import warnings
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import minmax_scale
from sklearn.manifold import trustworthiness
import matplotlib.pyplot as plt

try:
    import hdbscan
except Exception as e:
    hdbscan = None
    print("HDBSCAN unavailable:", e)

try:
    import umap
except Exception as e:
    umap = None
    print("UMAP unavailable:", e)

from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

def gpu_report():
    report = {
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }
    if torch.cuda.is_available():
        idx = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(idx)
        report.update({
            "gpu_name": props.name,
            "total_vram_gb": round(props.total_memory / 1024**3, 2),
            "allocated_gb": round(torch.cuda.memory_allocated(idx) / 1024**3, 3),
            "reserved_gb": round(torch.cuda.memory_reserved(idx) / 1024**3, 3),
        })
    return report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if (DEVICE == "cuda" and USE_FP16_ON_CUDA) else torch.float32
print(json.dumps(gpu_report(), indent=2))
print("Selected dtype:", DTYPE)

In [ ]:
# =============================
# 4. Colab upload helper and bundle verification
# =============================
def maybe_upload_bundle_for_colab():
    # In Colab, allow manual upload if the bundle path is not present.
    if BUNDLE_PATH.exists():
        return
    # try:
    #     import google.colab  # type: ignore
    #     from google.colab import files  # type: ignore
    #     print("Bundle not found. Upload clinical_cases_bundle.duckdb now.")
    #     uploaded = files.upload()
    #     for name in uploaded:
    #         if name.endswith(".duckdb"):
    #             shutil.move(name, BUNDLE_PATH.name)
    #             BUNDLE_PATH = Path(BUNDLE_PATH.name)
    #             print("Uploaded bundle:", BUNDLE_PATH)
    #             return
    # except Exception:
    #     pass
    # raise FileNotFoundError(f"Bundle not found: {BUNDLE_PATH}")

maybe_upload_bundle_for_colab()
assert BUNDLE_PATH.exists(), BUNDLE_PATH

def duckdb_table_counts(con):
    tables = con.execute("SHOW TABLES").fetchdf()["name"].tolist()
    counts = {}
    for t in tables:
        counts[t] = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    return pd.DataFrame([{"table": k, "row_count": v} for k, v in counts.items()]).sort_values("table")

con = duckdb.connect(str(BUNDLE_PATH), read_only=True)
required_tables = {"cases", "pages", "source_lineage", "acceptance", "embeddings", "clusters", "star_case_scores"}
actual_tables = set(con.execute("SHOW TABLES").fetchdf()["name"].tolist())
missing = required_tables - actual_tables
assert not missing, missing
counts_df = duckdb_table_counts(con)
display(counts_df)

cases_count = con.execute("SELECT COUNT(*) FROM cases").fetchone()[0]
lineage_count = con.execute("SELECT COUNT(*) FROM source_lineage").fetchone()[0]
acceptance_count = con.execute("SELECT COUNT(*) FROM acceptance").fetchone()[0]
preface_count = con.execute("SELECT COUNT(*) FROM cases WHERE case_id = 'prefacio_27_28'").fetchone()[0]
acceptance_df = con.execute("SELECT * FROM acceptance").fetchdf()

# ----
assert acceptance_count == 1, acceptance_count
assert lineage_count == cases_count, (lineage_count, cases_count)

status = acceptance_df["acceptance_status"].iloc[0]
print("Acceptance status:", status)

assert status in {
    "ACCEPTED_CLEAN_CANONICAL_BASELINE",
    "ACCEPTED_SPLIT_BASELINE",
    "PENDING_SPLIT_QA_BASELINE",
}, status


# ----
print("Base bundle verified.")
display(acceptance_df)
con.close()

# =============================
# 5. Load cases, pages, and lineage
# =============================
con = duckdb.connect(str(BUNDLE_PATH), read_only=True)
cases_df = con.execute("""
SELECT
  case_id, title, section, subsection,
  printed_start_page, printed_end_page,
  page_count, char_count, clean_text,
  source_pdf_path, clean_pdf_path,
  boundary_decision, boundary_source, ocr_version
FROM cases
ORDER BY printed_start_page, case_id
""").fetchdf()
pages_df = con.execute("SELECT * FROM pages ORDER BY case_id, page_number").fetchdf()
lineage_df = con.execute("SELECT * FROM source_lineage ORDER BY case_id").fetchdf()
con.close()

assert cases_df["case_id"].is_unique
assert cases_df["clean_text"].notna().all()
assert (cases_df["clean_text"].str.len() > 0).all()

cases_df["clean_text_len"] = cases_df["clean_text"].str.len()
cases_df["char_count_delta"] = (cases_df["char_count"] - cases_df["clean_text_len"]).abs()
print(cases_df[["case_id", "title", "section", "subsection", "char_count", "clean_text_len"]].head())
print(cases_df["clean_text_len"].describe())


# ---------------

# =============================
# 5B. Select working cases
# =============================

def select_work_cases(cases_df, run_mode="smoke", smoke_n_cases=20):
    """
    Book-agnostic case selection.

    - full: use all cases.
    - smoke: select a small, section-balanced subset.
    """
    if run_mode != "smoke":
        return cases_df.copy()

    # Prefer section-balanced smoke sample.
    sort_cols = ["section", "printed_start_page", "case_id"]
    available_sort_cols = [c for c in sort_cols if c in cases_df.columns]

    smoke_df = (
        cases_df
        .sort_values(available_sort_cols)
        .groupby("section", group_keys=False)
        .head(2)
        .head(smoke_n_cases)
        .copy()
    )

    # Fallback in case section is missing or grouping returns too little.
    if smoke_df.empty:
        smoke_df = (
            cases_df
            .sort_values(["printed_start_page", "case_id"])
            .head(smoke_n_cases)
            .copy()
        )

    return smoke_df


work_cases_df = select_work_cases(
    cases_df=cases_df,
    run_mode=RUN_MODE,
    smoke_n_cases=SMOKE_N_CASES,
)

assert not work_cases_df.empty
assert work_cases_df["case_id"].is_unique

print("Working cases:", len(work_cases_df), "of", len(cases_df))
display(work_cases_df[["case_id", "title", "section", "subsection", "char_count"]].head(20))

# ----------------------------------------------


assert acceptance_count == 1, acceptance_count
assert lineage_count == cases_count, (lineage_count, cases_count)

status = acceptance_df["acceptance_status"].iloc[0]
print("Acceptance status:", status)

assert status in {
    "ACCEPTED_CLEAN_CANONICAL_BASELINE",
    "ACCEPTED_SPLIT_BASELINE",
    "PENDING_SPLIT_QA_BASELINE",
}, status



In [ ]:
# =============================
# 6. Full-text case document construction and token chunking
# =============================
def build_case_document(row):
    meta = [
        f"Título: {row['title']}",
        f"Sección: {row['section']}",
        f"Subsección: {row['subsection']}",
        f"Páginas impresas: {row['printed_start_page']}–{row['printed_end_page']}",
        "Texto clínico OCR limpio:",
    ]
    return "\n".join(meta) + "\n" + str(row["clean_text"])

# Use tokenizer of first selected embedding model for chunking; fallback to whitespace if loading fails.
def load_chunk_tokenizer():
    if SELECTED_EMBEDDING_KEYS:
        key = SELECTED_EMBEDDING_KEYS[0]
        model_id = EMBEDDING_MODEL_REGISTRY[key]["model_id"]
        try:
            tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=TRUST_REMOTE_CODE)
            print("Chunk tokenizer:", model_id)
            return tok
        except Exception as e:
            print("Tokenizer load failed, using baseline tokenizer. Error:", repr(e))
    try:
        return AutoTokenizer.from_pretrained("BAAI/bge-m3", trust_remote_code=TRUST_REMOTE_CODE)
    except Exception:
        return None

chunk_tokenizer = load_chunk_tokenizer()

def chunk_text_by_tokens(text, tokenizer, chunk_tokens=768, overlap=128):
    if tokenizer is None:
        # Conservative whitespace fallback.
        words = text.split()
        chunks = []
        start = 0
        while start < len(words):
            end = min(start + chunk_tokens, len(words))
            chunk = " ".join(words[start:end])
            chunks.append((chunk, start, end))
            if end == len(words):
                break
            start = max(0, end - overlap)
        return chunks

    token_ids = tokenizer.encode(text, add_special_tokens=False)
    if len(token_ids) <= chunk_tokens:
        return [(text, 0, len(token_ids))]
    chunks = []
    start = 0
    while start < len(token_ids):
        end = min(start + chunk_tokens, len(token_ids))
        ids = token_ids[start:end]
        chunk = tokenizer.decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        chunks.append((chunk, start, end))
        if end >= len(token_ids):
            break
        start = max(0, end - overlap)
    return chunks

chunk_records = []
for _, row in tqdm(work_cases_df.iterrows(), total=len(work_cases_df), desc="Chunking cases"):
    doc = build_case_document(row)
    raw_chunks = chunk_text_by_tokens(doc, chunk_tokenizer, CHUNK_TOKENS, CHUNK_OVERLAP)
    if SMOKE_LIMIT_CHUNKS is not None and RUN_MODE == "smoke":
        raw_chunks = raw_chunks[:SMOKE_LIMIT_CHUNKS]
    filtered = []
    for ctext, t0, t1 in raw_chunks:
        if len(ctext) >= MIN_CHUNK_CHARS or len(raw_chunks) == 1:
            filtered.append((ctext, t0, t1))
    for idx, (ctext, t0, t1) in enumerate(filtered):
        chunk_records.append({
            "case_id": row["case_id"],
            "chunk_id": f"{row['case_id']}::chunk_{idx:04d}",
            "chunk_index": idx,
            "chunk_text": ctext,
            "chunk_char_count": len(ctext),
            "token_start": int(t0),
            "token_end": int(t1),
            "title": row["title"],
            "section": row["section"],
            "subsection": row["subsection"],
        })

chunks_df = pd.DataFrame(chunk_records)
assert not chunks_df.empty
chunks_per_case = chunks_df.groupby("case_id").size()
print("Cases:", len(work_cases_df))
print("Chunks:", len(chunks_df))
print("Chunks per case min/median/max:", chunks_per_case.min(), chunks_per_case.median(), chunks_per_case.max())
print("Total chunk chars:", int(chunks_df["chunk_char_count"].sum()))
display(chunks_per_case.describe())
display(chunks_df.head())

In [ ]:
# =============================
# 7. Embedding helpers
# =============================
def cleanup_torch():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def l2_normalize_np(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked = last_hidden_state * mask
    summed = masked.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def add_instruction(texts, instruction):
    return [instruction + t for t in texts]

def embed_with_transformers_mean_pool(texts, model_id, instruction="", batch_size=BATCH_SIZE):
    cleanup_torch()
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=TRUST_REMOTE_CODE)
    model = AutoModel.from_pretrained(
        model_id,
        trust_remote_code=TRUST_REMOTE_CODE,
        torch_dtype=DTYPE if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    model.eval()
    texts = add_instruction(texts, instruction)
    vectors = []
    active_bs = batch_size
    i = 0
    with torch.no_grad():
        while i < len(texts):
            batch = texts[i:i+active_bs]
            try:
                enc = tokenizer(batch, padding=True, truncation=True, max_length=CHUNK_TOKENS + 64, return_tensors="pt")
                enc = {k: v.to(model.device) for k, v in enc.items()}
                out = model(**enc)
                pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
                pooled = F.normalize(pooled, p=2, dim=1)
                vectors.append(pooled.detach().cpu().float().numpy())
                i += active_bs
            except torch.cuda.OutOfMemoryError:
                cleanup_torch()
                if active_bs <= 1:
                    raise
                active_bs = max(1, active_bs // 2)
                print("CUDA OOM. Reducing batch size to", active_bs)
    del model
    cleanup_torch()
    return np.vstack(vectors).astype(np.float32)

def embed_with_sentence_transformers(texts, model_id, instruction="", batch_size=BATCH_SIZE):
    cleanup_torch()
    try:
        model = SentenceTransformer(model_id, trust_remote_code=TRUST_REMOTE_CODE, device=DEVICE)
    except TypeError:
        model = SentenceTransformer(model_id, device=DEVICE)
    if hasattr(model, "max_seq_length"):
        try:
            model.max_seq_length = max(model.max_seq_length, CHUNK_TOKENS)
        except Exception:
            pass
    texts = add_instruction(texts, instruction)
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    del model
    cleanup_torch()
    return emb

def embed_chunks_for_model(chunks_df, key):
    spec = EMBEDDING_MODEL_REGISTRY[key]
    model_id = spec["model_id"]
    backend = spec["backend"]
    instruction = spec.get("instruction", "")
    texts = chunks_df["chunk_text"].tolist()
    print(f"Embedding {len(texts)} chunks with {key}: {model_id} ({backend})")
    t0 = time.time()
    try:
        if backend == "transformers_mean_pool":
            emb = embed_with_transformers_mean_pool(texts, model_id, instruction, BATCH_SIZE)
        elif backend == "sentence_transformers":
            emb = embed_with_sentence_transformers(texts, model_id, instruction, BATCH_SIZE)
        elif backend == "sentence_transformers_or_transformers":
            try:
                emb = embed_with_sentence_transformers(texts, model_id, instruction, BATCH_SIZE)
            except Exception as e1:
                print("SentenceTransformer path failed; trying transformers mean pooling:", repr(e1))
                emb = embed_with_transformers_mean_pool(texts, model_id, instruction, BATCH_SIZE)
        else:
            raise ValueError(f"Unknown backend: {backend}")
        status = "success"
        error = None
    except Exception as e:
        emb = None
        status = "failed"
        error = repr(e)
    elapsed = time.time() - t0
    return emb, {
        "model_key": key,
        "model_id": model_id,
        "backend": backend,
        "status": status,
        "error": error,
        "elapsed_s": elapsed,
        "n_chunks": len(texts),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

In [ ]:
# =============================
# 7. Run chunk embeddings and pool to case embeddings
# =============================
if not RUN_SOTA:
    raise RuntimeError("RUN_SOTA=False is not supported in this unified notebook. Keep RUN_SOTA=True to build the final bundle.")

all_chunk_embedding_rows = []
all_case_embedding_rows = []
all_embedding_run_rows = []
model_vectors = {}

for key in SELECTED_EMBEDDING_KEYS:
    emb, meta = embed_chunks_for_model(chunks_df, key)
    all_embedding_run_rows.append(meta);
    if emb is None:
        print(f"FAILED {key}: {meta['error']}")
        if not ALLOW_MODEL_FALLBACK:
            raise RuntimeError(meta["error"])
        continue

    model_vectors[key] = emb
    chunk_meta = chunks_df[["case_id", "chunk_id", "chunk_index", "chunk_text", "chunk_char_count"]].reset_index(drop=True);

    for idx, row in chunk_meta.iterrows():
        all_chunk_embedding_rows.append({
            "case_id": row["case_id"],
            "chunk_id": row["chunk_id"],
            "chunk_index": int(row["chunk_index"]),
            "model_key": key,
            "embedding_model": EMBEDDING_MODEL_REGISTRY[key]["model_id"],
            "embedding_dim": int(emb.shape[1]),
            "embedding_vector_json": json.dumps(emb[idx].astype(float).tolist()),
            "chunk_chars": int(row["chunk_char_count"]),
            "created_at": meta["created_at"],
        })

    # Mean pool chunks to one case vector per model.
    for case_id, idxs in chunk_meta.groupby("case_id").indices.items():
        pooled = emb[list(idxs)].mean(axis=0, keepdims=True)
        pooled = l2_normalize_np(pooled)[0]
        all_case_embedding_rows.append({
            "case_id": case_id,
            "model_key": key,
            "embedding_model": EMBEDDING_MODEL_REGISTRY[key]["model_id"],
            "embedding_dim": int(pooled.shape[0]),
            "embedding_vector_json": json.dumps(pooled.astype(float).tolist()),
            "n_chunks": int(len(idxs)),
            "created_at": meta["created_at"],
        })

chunk_embeddings_df = pd.DataFrame(all_chunk_embedding_rows)
case_embeddings_df = pd.DataFrame(all_case_embedding_rows)
embedding_runs_df = pd.DataFrame(all_embedding_run_rows)

successful_keys = sorted(case_embeddings_df["model_key"].unique().tolist()) if not case_embeddings_df.empty else []
assert successful_keys, "No embedding models succeeded."

print("Successful embedding models:", successful_keys)
display(embedding_runs_df)
display(case_embeddings_df.groupby("model_key").agg(n_cases=("case_id", "count"), dim=("embedding_dim", "first")))

In [ ]:
# =============================
# 9. Case-neighbor analysis and model consensus
# =============================
def parse_vecs(df):
    return np.vstack(df["embedding_vector_json"].map(lambda s: np.array(json.loads(s), dtype=np.float32)).to_list())

nearest_rows = []
case_model_matrices = {}
for key in case_embeddings_df["model_key"].unique():
    sub = case_embeddings_df[case_embeddings_df.model_key == key].sort_values("case_id").reset_index(drop=True)
    case_ids = sub["case_id"].tolist()
    X = parse_vecs(sub)
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, -np.inf)
    case_model_matrices[key] = {"case_ids": case_ids, "X": X, "sim": sim}
    meta = cases_df.set_index("case_id")
    for i, case_id in enumerate(case_ids):
        top_idx = np.argsort(-sim[i])[:NEIGHBOR_TOP_K]
        for rank, j in enumerate(top_idx, start=1):
            neighbor_id = case_ids[j]
            nearest_rows.append({
                "model_key": key,
                "embedding_model": EMBEDDING_MODEL_REGISTRY[key]["model_id"],
                "case_id": case_id,
                "neighbor_case_id": neighbor_id,
                "rank": rank,
                "similarity": float(sim[i, j]),
                "case_title": meta.loc[case_id, "title"],
                "neighbor_title": meta.loc[neighbor_id, "title"],
                "case_section": meta.loc[case_id, "section"],
                "neighbor_section": meta.loc[neighbor_id, "section"],
                "case_subsection": meta.loc[case_id, "subsection"],
                "neighbor_subsection": meta.loc[neighbor_id, "subsection"],
                "same_section": bool(meta.loc[case_id, "section"] == meta.loc[neighbor_id, "section"]),
                "same_subsection": bool(meta.loc[case_id, "subsection"] == meta.loc[neighbor_id, "subsection"]),
            })
nearest_neighbors_df = pd.DataFrame(nearest_rows)

def reciprocal_rank_weight(rank):
    return 1.0 / rank

consensus = nearest_neighbors_df.copy()
consensus["rr_weight"] = consensus["rank"].map(reciprocal_rank_weight)
consensus_neighbors_df = consensus.groupby(["case_id", "neighbor_case_id"], as_index=False).agg(
    n_models=("model_key", "nunique"),
    consensus_score=("rr_weight", "sum"),
    mean_similarity=("similarity", "mean"),
    best_rank=("rank", "min"),
    same_section=("same_section", "first"),
    same_subsection=("same_subsection", "first"),
)
consensus_neighbors_df = consensus_neighbors_df.sort_values(["case_id", "consensus_score"], ascending=[True, False])
consensus_neighbors_df["consensus_rank"] = consensus_neighbors_df.groupby("case_id").cumcount() + 1
consensus_neighbors_df = consensus_neighbors_df[consensus_neighbors_df.consensus_rank <= NEIGHBOR_TOP_K]

print("Nearest neighbors rows:", len(nearest_neighbors_df))
print("Consensus neighbors rows:", len(consensus_neighbors_df))
display(consensus_neighbors_df.head(20))

In [ ]:
# =============================
# 10. Clustering, model sweep, and stability metrics
# =============================
def cluster_quality(X, labels):
    labels = np.asarray(labels)
    valid = labels >= 0
    n_clusters = len(set(labels[valid])) if valid.any() else 0
    noise_fraction = float((labels < 0).mean())
    result = {
        "n_clusters": int(n_clusters),
        "noise_fraction": noise_fraction,
        "min_cluster_size": None,
        "max_cluster_size": None,
        "silhouette_score": None,
        "davies_bouldin_score": None,
        "calinski_harabasz_score": None,
    }
    if n_clusters > 0:
        sizes = pd.Series(labels[valid]).value_counts()
        result["min_cluster_size"] = int(sizes.min())
        result["max_cluster_size"] = int(sizes.max())
    if n_clusters >= 2 and valid.sum() > n_clusters:
        try:
            result["silhouette_score"] = float(silhouette_score(X[valid], labels[valid], metric="cosine"))
        except Exception:
            pass
        try:
            result["davies_bouldin_score"] = float(davies_bouldin_score(X[valid], labels[valid]))
        except Exception:
            pass
        try:
            result["calinski_harabasz_score"] = float(calinski_harabasz_score(X[valid], labels[valid]))
        except Exception:
            pass
    return result

cluster_report_rows = []
cluster_label_store = {}

for key, bundle in case_model_matrices.items():
    X = bundle["X"]
    case_ids = bundle["case_ids"]
    # HDBSCAN
    if hdbscan is not None and len(X) >= 10:
        for min_cluster_size in [3, 5, 8, 12]:
            try:
                h = hdbscan.HDBSCAN(min_cluster_size=min_cluster_size, metric="euclidean", prediction_data=False)
                labels = h.fit_predict(X)
                qual = cluster_quality(X, labels)
                row = {"model_key": key, "method": "hdbscan", "k": None, "min_cluster_size_param": min_cluster_size, **qual}
                cluster_report_rows.append(row)
                cluster_label_store[(key, "hdbscan", str(min_cluster_size))] = labels
            except Exception as e:
                cluster_report_rows.append({"model_key": key, "method": "hdbscan", "k": None, "min_cluster_size_param": min_cluster_size, "error": repr(e)})
    # KMeans
    for k in KMEANS_K_GRID:
        if k >= len(X):
            continue
        labels = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init="auto").fit_predict(X)
        qual = cluster_quality(X, labels)
        row = {"model_key": key, "method": "kmeans", "k": k, "min_cluster_size_param": None, **qual}
        cluster_report_rows.append(row)
        cluster_label_store[(key, "kmeans", str(k))] = labels
    # Agglomerative
    for k in AGGLOMERATIVE_K_GRID:
        if k >= len(X):
            continue
        labels = AgglomerativeClustering(n_clusters=k, metric="cosine", linkage="average").fit_predict(X)
        qual = cluster_quality(X, labels)
        row = {"model_key": key, "method": "agglomerative", "k": k, "min_cluster_size_param": None, **qual}
        cluster_report_rows.append(row)
        cluster_label_store[(key, "agglomerative", str(k))] = labels

cluster_model_report_df = pd.DataFrame(cluster_report_rows)
# Selection heuristic: prefer valid silhouette, not all noise, max reasonable score.
selectable = cluster_model_report_df.dropna(subset=["silhouette_score"]).copy()
selectable = selectable[(selectable["n_clusters"] >= 3) & (selectable["noise_fraction"] < 0.5)]
if selectable.empty:
    selected_row = cluster_model_report_df.dropna(subset=["n_clusters"]).iloc[0].to_dict()
else:
    selected_row = selectable.sort_values(["silhouette_score", "n_clusters"], ascending=[False, True]).iloc[0].to_dict()

cluster_model_report_df["selected"] = False
mask = (
    (cluster_model_report_df["model_key"] == selected_row["model_key"]) &
    (cluster_model_report_df["method"] == selected_row["method"]) &
    (cluster_model_report_df["k"].fillna(-1) == (selected_row.get("k") if pd.notna(selected_row.get("k")) else -1)) &
    (cluster_model_report_df["min_cluster_size_param"].fillna(-1) == (selected_row.get("min_cluster_size_param") if pd.notna(selected_row.get("min_cluster_size_param")) else -1))
)
cluster_model_report_df.loc[mask, "selected"] = True

selected_key = selected_row["model_key"]
selected_method = selected_row["method"]
selected_param = str(int(selected_row["k"])) if pd.notna(selected_row.get("k")) else str(int(selected_row.get("min_cluster_size_param") or 0))
selected_labels = cluster_label_store[(selected_key, selected_method, selected_param)]
print("Selected cluster model:", selected_key, selected_method, selected_param)
display(cluster_model_report_df.sort_values("silhouette_score", ascending=False).head(15))

In [ ]:
# =============================
# 11. Projection sweep: UMAP + optional PaCMAP/TriMAP
# =============================
projection_rows = []
projection_quality_rows = []
selected_X = case_model_matrices[selected_key]["X"]
selected_case_ids = case_model_matrices[selected_key]["case_ids"]

if umap is None:
    raise RuntimeError("umap-learn is required for projections.")

for n_neighbors in UMAP_NEIGHBORS_GRID:
    if n_neighbors >= len(selected_X):
        continue
    for min_dist in UMAP_MIN_DIST_GRID:
        for seed in PROJECTION_SEEDS:
            try:
                reducer = umap.UMAP(
                    n_neighbors=n_neighbors,
                    min_dist=min_dist,
                    metric="cosine",
                    random_state=seed,
                    n_components=2,
                )
                coords = reducer.fit_transform(selected_X)
                tw = trustworthiness(selected_X, coords, n_neighbors=min(10, len(selected_X)-1), metric="cosine")
                # neighbor preservation@10
                sim_hi = cosine_similarity(selected_X)
                np.fill_diagonal(sim_hi, -np.inf)
                high_nn = np.argsort(-sim_hi, axis=1)[:, :min(10, len(selected_X)-1)]
                dist_lo = ((coords[:, None, :] - coords[None, :, :]) ** 2).sum(axis=2)
                np.fill_diagonal(dist_lo, np.inf)
                low_nn = np.argsort(dist_lo, axis=1)[:, :min(10, len(selected_X)-1)]
                preservation = np.mean([len(set(high_nn[i]).intersection(set(low_nn[i]))) / high_nn.shape[1] for i in range(len(selected_X))])
                run_id = f"umap_nn{n_neighbors}_md{min_dist}_seed{seed}"
                projection_quality_rows.append({
                    "projection_run_id": run_id,
                    "method": "umap",
                    "n_neighbors": n_neighbors,
                    "min_dist": min_dist,
                    "seed": seed,
                    "trustworthiness": float(tw),
                    "neighbor_preservation_at_10": float(preservation),
                })
                for i, cid in enumerate(selected_case_ids):
                    projection_rows.append({
                        "projection_run_id": run_id,
                        "case_id": cid,
                        "method": "umap",
                        "x": float(coords[i, 0]),
                        "y": float(coords[i, 1]),
                        "n_neighbors": n_neighbors,
                        "min_dist": min_dist,
                        "seed": seed,
                    })
            except Exception as e:
                projection_quality_rows.append({
                    "projection_run_id": f"umap_nn{n_neighbors}_md{min_dist}_seed{seed}",
                    "method": "umap", "n_neighbors": n_neighbors, "min_dist": min_dist, "seed": seed,
                    "error": repr(e),
                })

projection_quality_df = pd.DataFrame(projection_quality_rows)
umap_coordinates_df = pd.DataFrame(projection_rows)
selected_projection = projection_quality_df.dropna(subset=["trustworthiness", "neighbor_preservation_at_10"]).copy()
selected_projection["projection_score"] = 0.6 * selected_projection["trustworthiness"] + 0.4 * selected_projection["neighbor_preservation_at_10"]
best_projection_row = selected_projection.sort_values("projection_score", ascending=False).iloc[0]
best_projection_id = best_projection_row["projection_run_id"]
best_coords = umap_coordinates_df[umap_coordinates_df.projection_run_id == best_projection_id].copy()
print("Best projection:", best_projection_id)
display(projection_quality_df.sort_values(["trustworthiness", "neighbor_preservation_at_10"], ascending=False).head(10))

# Attach best UMAP coords to selected clusters.
clusters_df = pd.DataFrame({
    "case_id": selected_case_ids,
    "embedding_model": EMBEDDING_MODEL_REGISTRY[selected_key]["model_id"],
    "model_key": selected_key,
    "cluster_method": f"{selected_method}:{selected_param}",
    "cluster_id": selected_labels.astype(int),
    "created_at": datetime.now(timezone.utc).isoformat(),
})
clusters_df = clusters_df.merge(best_coords[["case_id", "x", "y"]], on="case_id", how="left").rename(columns={"x": "umap_x", "y": "umap_y"})
clusters_df["outlier_score"] = np.where(clusters_df["cluster_id"] < 0, 1.0, 0.0)
clusters_df["silhouette_score"] = selected_row.get("silhouette_score")

display(clusters_df.head())

In [ ]:
import re
# =============================
# 12. Build analysis table for scoring and optional plots
# =============================
plot_df = clusters_df.merge(
    cases_df[["case_id", "title", "section", "subsection", "char_count"]],
    on="case_id",
    how="left",
)

SECTION_NAME_OVERRIDES = {
    "seccion1": "1.- Química clínica, equilibrio ácido-base y líquidos biológicos",
    "seccion2": "2.- Hematología, anemias y hemostasia",
    "seccion3": "3.- Inmunología clínica e hipersensibilidad",
    "seccion4": "4.- Microbiología bacteriana y tracto urinario",
    "seccion5": "5.- Infectología clínica",
    "seccion6": "6.- Parasitología y artrópodos",
}

def humanize_section_code(section):
    text = str(section)
    if text in SECTION_NAME_OVERRIDES:
        return SECTION_NAME_OVERRIDES[text]
    m = re.search(r"(\d+)", text)
    if m:
        return f"{int(m.group(1))}.- {text}"
    return text

plot_df["section_label"] = plot_df["section"].map(humanize_section_code)

if ENABLE_PLOTS:
    fig, ax = plt.subplots(figsize=(10, 7))
    for section_label, g in plot_df.groupby("section_label", sort=True):
        ax.scatter(g["umap_x"], g["umap_y"], label=section_label, s=62, alpha=0.78)
    ax.set_title("Best UMAP projection by section")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.legend(bbox_to_anchor=(1.04, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()

display(plot_df[["case_id", "section", "subsection", "cluster_id", "umap_x", "umap_y"]].head())


In [ ]:
# =============================
# 13. Topic terms by selected cluster
# =============================
import re
import unicodedata
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer

def strip_accents(text):
    return "".join(
        c for c in unicodedata.normalize("NFKD", str(text))
        if not unicodedata.combining(c)
    )

def normalize_term_text(text):
    text = strip_accents(str(text).lower())
    text = re.sub(r"\b\d+(?:[.,]\d+)?\b", " ", text)
    text = re.sub(r"[^a-zñü\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

SPANISH_STOPWORDS = set("""
a al algo algunas algunos ante antes aqui como con contra cual cuando de del desde donde dos el ella ellas ellos en entre era
eran es esa esas ese eso esos esta estaba estan este esto estos fue fueron ha han hay la las le les lo los mas me mi mis muy
no o para pero por que se si sin sobre son su sus tambien tiene un una unas unos y ya
""".split())

DOMAIN_NOISE = set("""
caso problema paciente pacientes examen examenes muestra muestras resultado resultados laboratorio clinico clinica tecnico medica
pregunta preguntas respuesta respuestas comentarios adicionales lectura sugerida pagina paginas capitulo seccion subseccion
""".split())

STOPWORDS = SPANISH_STOPWORDS | DOMAIN_NOISE

def case_text_for_topics(row):
    return normalize_term_text(
        " ".join([
            str(row.get("title", "")),
            str(row.get("section", "")),
            str(row.get("subsection", "")),
            str(row.get("clean_text", "")),
        ])
    )

topic_source_df = plot_df[["case_id", "cluster_id"]].merge(
    cases_df[["case_id", "title", "section", "subsection", "clean_text"]],
    on="case_id",
    how="left",
)

cluster_docs = []
cluster_ids = []
for cluster_id, g in topic_source_df.groupby("cluster_id"):
    if int(cluster_id) < 0:
        continue
    cluster_ids.append(int(cluster_id))
    cluster_docs.append(" ".join(g.apply(case_text_for_topics, axis=1).tolist()))

topic_rows = []
topic_term_audit_rows = []

if cluster_docs:
    vectorizer = CountVectorizer(
        stop_words=list(STOPWORDS),
        ngram_range=(1, 2),
        min_df=1,
        max_features=4000,
        token_pattern=r"(?u)\b[a-zñü][a-zñü]{2,}\b",
    )
    X_counts = vectorizer.fit_transform(cluster_docs)
    X_tfidf = TfidfTransformer(norm=None).fit_transform(X_counts)
    terms = np.array(vectorizer.get_feature_names_out())

    for row_idx, cluster_id in enumerate(cluster_ids):
        scores = np.asarray(X_tfidf[row_idx].todense()).ravel()
        order = np.argsort(-scores)
        rank = 1
        for j in order:
            term = terms[j]
            score = float(scores[j])
            if score <= 0:
                continue
            if len(term) < 3:
                continue
            topic_rows.append({
                "cluster_id": int(cluster_id),
                "rank": int(rank),
                "term": str(term),
                "score": score,
                "created_at": datetime.now(timezone.utc).isoformat(),
            })
            topic_term_audit_rows.append({
                "cluster_id": int(cluster_id),
                "term": str(term),
                "score": score,
                "accepted": True,
                "reason": "tfidf_cluster_term",
                "created_at": datetime.now(timezone.utc).isoformat(),
            })
            rank += 1
            if rank > 12:
                break

topic_terms_df = pd.DataFrame(topic_rows)
topic_term_audit_df = pd.DataFrame(topic_term_audit_rows)
topic_term_edges_df = pd.DataFrame()
topic_term_layout_df = pd.DataFrame()

display(topic_terms_df.head(30))


In [ ]:
# =============================
# 14. Diversity, coverage, and star-case scoring
# =============================
meta = cases_df.set_index("case_id")
# Use consensus neighbors for robust cross-neighborhood metrics.
neighbor_metrics = consensus_neighbors_df.groupby("case_id").agg(
    neighbor_diversity_score=("same_section", lambda x: 1.0 - float(np.mean(x))),
    cross_section_neighbor_rate=("same_section", lambda x: 1.0 - float(np.mean(x))),
    same_subsection_neighbor_rate=("same_subsection", lambda x: float(np.mean(x))),
    mean_consensus_score=("consensus_score", "mean"),
    max_consensus_score=("consensus_score", "max"),
).reset_index()

metrics_df = plot_df[["case_id", "section", "subsection", "cluster_id", "char_count"]].copy()
section_counts = metrics_df["section"].value_counts()
subsection_counts = metrics_df["subsection"].value_counts()
cluster_counts = metrics_df["cluster_id"].value_counts()
metrics_df["section_rarity_score"] = metrics_df["section"].map(lambda x: 1.0 / section_counts[x])
metrics_df["subsection_rarity_score"] = metrics_df["subsection"].map(lambda x: 1.0 / subsection_counts[x])
metrics_df["cluster_rarity_score"] = metrics_df["cluster_id"].map(lambda x: 1.0 / cluster_counts[x])
for col in ["section_rarity_score", "subsection_rarity_score", "cluster_rarity_score"]:
    metrics_df[col] = minmax_scale(metrics_df[col].to_numpy()) if metrics_df[col].nunique() > 1 else 0.5

# Length balance: reward cases near median length, penalize extremes.
lengths = metrics_df["char_count"].astype(float).to_numpy()
median_len = np.median(lengths)
mad = np.median(np.abs(lengths - median_len)) + 1e-9
z = np.abs(lengths - median_len) / (3 * mad)
metrics_df["length_balance_score"] = np.clip(1.0 - z, 0.0, 1.0)

metrics_df = metrics_df.merge(neighbor_metrics, on="case_id", how="left").fillna({
    "neighbor_diversity_score": 0.0,
    "cross_section_neighbor_rate": 0.0,
    "same_subsection_neighbor_rate": 0.0,
    "mean_consensus_score": 0.0,
    "max_consensus_score": 0.0,
})

# Semantic centrality/novelty from selected model similarities.
sel_sim = case_model_matrices[selected_key]["sim"]
sel_case_ids = case_model_matrices[selected_key]["case_ids"]
centrality_rows = []
for i, cid in enumerate(sel_case_ids):
    labels = selected_labels
    same = np.where((labels == labels[i]) & (np.arange(len(labels)) != i) & (labels >= 0))[0]
    top = np.argsort(-sel_sim[i])[:min(10, len(sel_sim)-1)]
    centrality = float(np.mean(sel_sim[i, same])) if len(same) else float(np.mean(sel_sim[i, top]))
    max_neighbor = float(np.max(sel_sim[i, top])) if len(top) else 0.0
    centrality_rows.append({
        "case_id": cid,
        "semantic_centrality_raw": centrality,
        "semantic_novelty_raw": 1.0 - max_neighbor,
    })
centrality_df = pd.DataFrame(centrality_rows)
for col in ["semantic_centrality_raw", "semantic_novelty_raw"]:
    vals = centrality_df[col].to_numpy()
    centrality_df[col.replace("_raw", "_score")] = minmax_scale(vals) if len(set(np.round(vals, 8))) > 1 else 0.5

metrics_df = metrics_df.merge(centrality_df, on="case_id", how="left")
metrics_df["curriculum_coverage_score"] = (
    0.35 * metrics_df["section_rarity_score"] +
    0.35 * metrics_df["subsection_rarity_score"] +
    0.30 * metrics_df["cluster_rarity_score"]
)
metrics_df["diversity_score"] = (
    0.25 * metrics_df["section_rarity_score"] +
    0.25 * metrics_df["subsection_rarity_score"] +
    0.20 * metrics_df["cluster_rarity_score"] +
    0.30 * metrics_df["neighbor_diversity_score"]
)

diversity_metrics_df = metrics_df.copy()

def classify_review(row):
    if row["teaching_score"] >= diversity_metrics_df["teaching_score"].quantile(0.80):
        return "high_priority_review"
    if row["semantic_centrality_score"] >= 0.70 and row["semantic_novelty_score"] < 0.70:
        return "representative_case"
    if row["semantic_novelty_score"] >= 0.80:
        return "unusual_case"
    if row["curriculum_coverage_score"] >= 0.75:
        return "curriculum_gap_case"
    return "candidate_for_review"

diversity_metrics_df["teaching_score"] = (
    0.25 * diversity_metrics_df["semantic_centrality_score"] +
    0.20 * diversity_metrics_df["diversity_score"] +
    0.20 * diversity_metrics_df["curriculum_coverage_score"] +
    0.15 * diversity_metrics_df["length_balance_score"] +
    0.10 * diversity_metrics_df["neighbor_diversity_score"] +
    0.10 * diversity_metrics_df["semantic_novelty_score"]
)
diversity_metrics_df["review_priority"] = diversity_metrics_df.apply(classify_review, axis=1)

star_case_scores_df = diversity_metrics_df.merge(cases_df[["case_id", "title", "section", "subsection"]], on=["case_id", "section", "subsection"], how="left")
star_case_scores_df["clarity_score"] = np.nan
star_case_scores_df["representativeness_score"] = star_case_scores_df["semantic_centrality_score"]
star_case_scores_df["novelty_score"] = star_case_scores_df["semantic_novelty_score"]
star_case_scores_df["recommended_use"] = star_case_scores_df["review_priority"]
star_case_scores_df["notes"] = "auto-scored; requires human review; not clinical advice"
star_case_scores_df["created_at"] = datetime.now(timezone.utc).isoformat()

star_cols = [
    "case_id", "title", "section", "subsection", "teaching_score", "diversity_score",
    "clarity_score", "representativeness_score", "novelty_score", "recommended_use", "notes", "created_at",
    "review_priority", "semantic_centrality_score", "curriculum_coverage_score", "length_balance_score", "neighbor_diversity_score",
]
star_case_scores_df = star_case_scores_df[star_cols].sort_values("teaching_score", ascending=False)

display(star_case_scores_df.head(25))

In [ ]:
# =============================
# 15. Select cases for GPT-OSS annotation
# =============================
def select_llm_cases(work_cases_df, star_case_scores_df=None):
    base = work_cases_df.copy()

    if (
        LLM_SELECTION_MODE == "star_first"
        and isinstance(star_case_scores_df, pd.DataFrame)
        and not star_case_scores_df.empty
        and "teaching_score" in star_case_scores_df.columns
    ):
        ordered_ids = (
            star_case_scores_df
            .sort_values(["teaching_score", "case_id"], ascending=[False, True])
            ["case_id"]
            .astype(str)
            .tolist()
        )
        order_df = pd.DataFrame({
            "case_id": ordered_ids,
            "_llm_order": range(len(ordered_ids)),
        })
        selected = (
            base.assign(case_id=base["case_id"].astype(str))
            .merge(order_df, on="case_id", how="left")
            .sort_values(["_llm_order", "case_id"], na_position="last")
            .drop(columns=["_llm_order"])
        )
    else:
        selected = base.sort_values("case_id").copy()

    if LLM_ANNOTATION_N_CASES is not None:
        selected = selected.head(int(LLM_ANNOTATION_N_CASES)).copy()

    return selected.reset_index(drop=True)

llm_cases_df = select_llm_cases(work_cases_df, star_case_scores_df)
print("LLM annotation cases:", len(llm_cases_df))
display(llm_cases_df[["case_id", "title", "section", "subsection"]].head(20))


In [ ]:
# =============================
# 6. Prompt and strict schema
# =============================
LLM_SCHEMA_KEYS = [
    "case_id",
    "clinical_area",
    "main_problem",
    "laboratory_methods",
    "key_concepts",
    "learning_objectives",
    "difficulty",
    "case_type",
    "star_case_score_llm",
    "star_case_rationale",
    "requires_human_review",
    "evidence_quotes",
]

LLM_SCHEMA_SPEC = {
    "case_id": "string copied exactly from input",
    "clinical_area": "short Spanish label",
    "main_problem": "one concise Spanish sentence",
    "laboratory_methods": ["method or exam mentioned in the case"],
    "key_concepts": ["short concept labels"],
    "learning_objectives": ["student-facing learning objective in Spanish"],
    "difficulty": "one of: baja, media, alta",
    "case_type": "one of: diagnostico, interpretacion_laboratorio, fisiopatologia, seguimiento, seguridad_paciente, otro",
    "star_case_score_llm": "number from 0.0 to 1.0; educational value as representative or high-yield case",
    "star_case_rationale": "brief Spanish rationale; no clinical advice",
    "requires_human_review": "boolean",
    "evidence_quotes": ["1-3 very short quoted fragments from the case text"],
}

JSON_OUTPUT_TEMPLATE = {
    "case_id": "",
    "clinical_area": "",
    "main_problem": "",
    "laboratory_methods": [],
    "key_concepts": [],
    "learning_objectives": [],
    "difficulty": "media",
    "case_type": "diagnostico",
    "star_case_score_llm": 0.5,
    "star_case_rationale": "",
    "requires_human_review": False,
    "evidence_quotes": [],
}

def build_case_document(row, max_chars=None):
    max_chars = LLM_MAX_INPUT_CHARS if max_chars is None else max_chars
    meta = [
        f"case_id: {row['case_id']}",
        f"Título: {row['title']}",
        f"Sección: {row['section']}",
        f"Subsección: {row['subsection']}",
        f"Páginas impresas: {row['printed_start_page']}–{row['printed_end_page']}",
        "Texto clínico OCR limpio:",
    ]
    full_text = "\n".join(meta) + "\n" + str(row["clean_text"])
    return full_text[:max_chars]

def build_annotation_prompt(row, max_chars=None, retry_mode=False):
    text = build_case_document(row, max_chars=max_chars)
    schema_text = json.dumps(LLM_SCHEMA_SPEC, ensure_ascii=False, indent=2)
    template_text = json.dumps(JSON_OUTPUT_TEMPLATE, ensure_ascii=False, indent=2)
    retry_line = (
        "This is a retry after invalid JSON. You must output only parseable JSON. "
        if retry_mode else ""
    )
    return f"""
You are extracting structured educational metadata from a Spanish clinical-laboratory teaching case.

{retry_line}Hard output rules:
- Output exactly one JSON object.
- Start with {{ and end with }}.
- Do not output Markdown.
- Do not output explanations.
- Do not output chain-of-thought.
- Do not wrap the JSON in code fences.
- Use double quotes for all JSON strings.
- Do not use trailing commas.
- Do not invent facts outside the case text.
- Use Spanish for human-facing fields.
- Copy case_id exactly.
- Keep evidence_quotes short and copied from the case text.
- star_case_score_llm means educational usefulness for selecting "casos estrella", not medical severity.

Required JSON schema:
{schema_text}

Use this exact object shape:
{template_text}

Case text:
{text}
""".strip()

def build_messages(row, max_chars=None, retry_mode=False):
    return [
        {
            "role": "system",
            "content": (
                "You extract structured educational metadata from Spanish clinical-laboratory "
                "teaching cases. Output exactly one compact valid JSON object only. "
                "No Markdown. No explanations. No chain-of-thought."
            ),
        },
        {
            "role": "user",
            "content": build_annotation_prompt(row, max_chars=max_chars, retry_mode=retry_mode),
        },
    ]

print(build_annotation_prompt(llm_cases_df.iloc[0])[:1800] if len(llm_cases_df) else 'No LLM cases selected.')


In [ ]:
# =============================
# 7. JSON parsing and validation helpers
# =============================
import ast

def strip_code_fences(text):
    text = str(text).strip()
    text = re.sub(r"^\s*```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    return text.strip()

def json_object_candidates(text):
    """Return all balanced {...} candidates from the model output."""
    text = strip_code_fences(text)
    candidates = []
    stack = []
    in_string = False
    escape = False
    start = None

    for i, ch in enumerate(text):
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            if not stack:
                start = i
            stack.append(ch)
        elif ch == "}":
            if stack:
                stack.pop()
                if not stack and start is not None:
                    candidates.append(text[start:i+1])
                    start = None
    return candidates

def parse_json_candidate(obj_text):
    """Try strict JSON first, then conservative repairs."""
    last_error = None

    for candidate in [obj_text, strip_code_fences(obj_text)]:
        try:
            return json.loads(candidate)
        except Exception as e:
            last_error = e

    # Optional repair package if available in the runtime.
    try:
        from json_repair import repair_json
        repaired = repair_json(obj_text)
        return json.loads(repaired)
    except Exception as e:
        last_error = e

    # Last fallback for Python-like dicts: single quotes, True/False/None.
    try:
        parsed = ast.literal_eval(obj_text)
        if isinstance(parsed, dict):
            return parsed
    except Exception as e:
        last_error = e

    raise last_error

def choose_best_json_object(raw_output):
    candidates = json_object_candidates(raw_output)
    if not candidates:
        raise ValueError("No complete JSON object found")

    parsed_candidates = []
    errors = []
    required = set(LLM_SCHEMA_KEYS)

    for obj_text in candidates:
        try:
            parsed = parse_json_candidate(obj_text)
            if isinstance(parsed, dict):
                score = len(required.intersection(parsed.keys()))
                parsed_candidates.append((score, len(obj_text), parsed, obj_text))
        except Exception as e:
            errors.append(repr(e))

    if not parsed_candidates:
        raise ValueError("JSON candidates found but none parsed: " + " | ".join(errors[:3]))

    # Prefer candidate with most required keys; if tied, prefer larger object.
    parsed_candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    _, _, parsed, obj_text = parsed_candidates[0]
    return parsed, obj_text

def normalize_list_value(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    if isinstance(x, str):
        # Keep a single string as one item; do not split medical phrases aggressively.
        return [x.strip()] if x.strip() else []
    return [str(x)]

def normalize_bool(x):
    if isinstance(x, bool):
        return x
    if isinstance(x, str):
        return x.strip().lower() in {"true", "sí", "si", "yes", "1", "verdadero"}
    return bool(x)

def normalize_score(x):
    try:
        value = float(x)
    except Exception:
        return None
    return max(0.0, min(1.0, value))

def normalize_enum(value, allowed, default):
    value = str(value or "").strip().lower()
    return value if value in allowed else default

def parse_and_normalize_llm_output(raw_output, expected_case_id):
    raw_output = str(raw_output)
    parsed, obj_text = choose_best_json_object(raw_output)

    missing = [k for k in LLM_SCHEMA_KEYS if k not in parsed]

    normalized = {k: parsed.get(k) for k in LLM_SCHEMA_KEYS}
    normalized["case_id"] = str(normalized.get("case_id") or expected_case_id)
    normalized["clinical_area"] = str(normalized.get("clinical_area") or "").strip()
    normalized["main_problem"] = str(normalized.get("main_problem") or "").strip()
    normalized["laboratory_methods"] = normalize_list_value(normalized.get("laboratory_methods"))
    normalized["key_concepts"] = normalize_list_value(normalized.get("key_concepts"))
    normalized["learning_objectives"] = normalize_list_value(normalized.get("learning_objectives"))
    normalized["evidence_quotes"] = normalize_list_value(normalized.get("evidence_quotes"))
    normalized["requires_human_review"] = normalize_bool(normalized.get("requires_human_review"))
    normalized["star_case_score_llm"] = normalize_score(normalized.get("star_case_score_llm"))
    normalized["difficulty"] = normalize_enum(
        normalized.get("difficulty"),
        {"baja", "media", "alta"},
        "media",
    )
    normalized["case_type"] = normalize_enum(
        normalized.get("case_type"),
        {"diagnostico", "interpretacion_laboratorio", "fisiopatologia", "seguimiento", "seguridad_paciente", "otro"},
        "otro",
    )
    normalized["star_case_rationale"] = str(normalized.get("star_case_rationale") or "").strip()

    if normalized["case_id"] != expected_case_id:
        missing.append("case_id_mismatch")

    return normalized, missing, obj_text

def repair_existing_llm_annotations_df(df):
    """Reparse raw_output in-place after parser improvements, without rerunning the LLM."""
    repaired_rows = []
    for _, r in df.iterrows():
        row = r.to_dict()
        try:
            parsed, missing_keys, json_object_text = parse_and_normalize_llm_output(
                row.get("raw_output", ""),
                expected_case_id=row["case_id"],
            )
            row["parsed_json"] = json.dumps(parsed, ensure_ascii=False)
            row["json_object_text"] = json_object_text
            row["json_parse_ok"] = True
            row["missing_keys_json"] = json.dumps(missing_keys, ensure_ascii=False)
            row["parse_error"] = None
            row.update({
                "clinical_area": parsed.get("clinical_area"),
                "main_problem": parsed.get("main_problem"),
                "laboratory_methods_json": json.dumps(parsed.get("laboratory_methods", []), ensure_ascii=False),
                "key_concepts_json": json.dumps(parsed.get("key_concepts", []), ensure_ascii=False),
                "learning_objectives_json": json.dumps(parsed.get("learning_objectives", []), ensure_ascii=False),
                "difficulty": parsed.get("difficulty"),
                "case_type": parsed.get("case_type"),
                "star_case_score_llm": parsed.get("star_case_score_llm"),
                "star_case_rationale": parsed.get("star_case_rationale"),
                "requires_human_review": parsed.get("requires_human_review"),
                "evidence_quotes_json": json.dumps(parsed.get("evidence_quotes", []), ensure_ascii=False),
            })
        except Exception as e:
            row["json_parse_ok"] = False
            row["parse_error"] = repr(e)
        repaired_rows.append(row)
    return pd.DataFrame(repaired_rows)

print("Robust parser ready.")


In [ ]:
# =============================
# 8. Load model and run annotations
# =============================
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_causal_lm(model_key):
    model_id = LLM_MODEL_REGISTRY[model_key]
    print("Loading local LLM:", model_key, model_id)

    tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True,
    )

    model_kwargs = {
        "trust_remote_code": True,
        "device_map": "auto" if DEVICE == "cuda" else None,
    }

    # transformers versions differ here. Try the newer dtype argument first, then fallback.
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype="auto",
            **model_kwargs,
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype="auto",
            **model_kwargs,
        )

    model.eval()
    return tok, model, model_id

def render_prompt(tokenizer, row, max_chars=None, retry_mode=False):
    messages = build_messages(row, max_chars=max_chars, retry_mode=retry_mode)
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return build_annotation_prompt(row, max_chars=max_chars, retry_mode=retry_mode)

def first_model_device(model):
    return next(model.parameters()).device

def generate_one_annotation(tokenizer, model, model_id, row, retry_mode=False):
    max_chars = LLM_RETRY_MAX_INPUT_CHARS if retry_mode else LLM_MAX_INPUT_CHARS
    prompt_text = render_prompt(tokenizer, row, max_chars=max_chars, retry_mode=retry_mode)
    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=LLM_MAX_PROMPT_TOKENS,
    )
    device = first_model_device(model)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    gen_kwargs = {
        "max_new_tokens": LLM_MAX_NEW_TOKENS,
        "do_sample": DO_SAMPLE,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if TEMPERATURE is not None:
        gen_kwargs["temperature"] = TEMPERATURE
    if TOP_P is not None:
        gen_kwargs["top_p"] = TOP_P

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    elapsed_s = time.time() - t0

    decoded = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    parsed = None
    json_object_text = None
    missing_keys = []
    parse_error = None

    try:
        parsed, missing_keys, json_object_text = parse_and_normalize_llm_output(
            decoded,
            expected_case_id=row["case_id"],
        )
    except Exception as e:
        parse_error = repr(e)

    base = {
        "case_id": row["case_id"],
        "title": row["title"],
        "section": row["section"],
        "subsection": row["subsection"],
        "model_key": LLM_MODEL_KEY,
        "model_name": model_id,
        "prompt_version": PROMPT_VERSION,
        "input_chars": min(len(build_case_document(row, max_chars=max_chars)), max_chars),
        "retry_mode": retry_mode,
        "raw_output": decoded,
        "parsed_json": json.dumps(parsed, ensure_ascii=False) if parsed else None,
        "json_object_text": json_object_text,
        "json_parse_ok": parsed is not None,
        "missing_keys_json": json.dumps(missing_keys, ensure_ascii=False),
        "parse_error": parse_error,
        "elapsed_s": elapsed_s,
        "output_chars": len(decoded),
        "created_at": datetime.now(timezone.utc).isoformat(),
    }

    if parsed:
        base.update({
            "clinical_area": parsed.get("clinical_area"),
            "main_problem": parsed.get("main_problem"),
            "laboratory_methods_json": json.dumps(parsed.get("laboratory_methods", []), ensure_ascii=False),
            "key_concepts_json": json.dumps(parsed.get("key_concepts", []), ensure_ascii=False),
            "learning_objectives_json": json.dumps(parsed.get("learning_objectives", []), ensure_ascii=False),
            "difficulty": parsed.get("difficulty"),
            "case_type": parsed.get("case_type"),
            "star_case_score_llm": parsed.get("star_case_score_llm"),
            "star_case_rationale": parsed.get("star_case_rationale"),
            "requires_human_review": parsed.get("requires_human_review"),
            "evidence_quotes_json": json.dumps(parsed.get("evidence_quotes", []), ensure_ascii=False),
        })

    return base

def run_llm_annotations(cases_subset):
    tokenizer, model, model_id = load_causal_lm(LLM_MODEL_KEY)

    rows = []

    if TEST_ONE_CASE_FIRST:
        print("Testing one case before batch execution...")
        first_row = generate_one_annotation(tokenizer, model, model_id, cases_subset.iloc[0])
        display(pd.DataFrame([{
            "case_id": first_row["case_id"],
            "json_parse_ok": first_row["json_parse_ok"],
            "parse_error": first_row["parse_error"],
            "star_case_score_llm": first_row.get("star_case_score_llm"),
            "elapsed_s": first_row["elapsed_s"],
        }]))
        assert first_row["json_parse_ok"], f"First-case LLM JSON parse failed: {first_row['parse_error']}\nRAW:\n{first_row['raw_output'][:2000]}"
        rows.append(first_row)
        iterator_df = cases_subset.iloc[1:].copy()
    else:
        iterator_df = cases_subset

    for _, row in tqdm(iterator_df.iterrows(), total=len(iterator_df), desc="LLM annotations"):
        rows.append(generate_one_annotation(tokenizer, model, model_id, row))

    # Reparse with the robust parser first; this can recover many valid-but-messy outputs.
    tmp_df = repair_existing_llm_annotations_df(pd.DataFrame(rows))
    failed_ids = set(tmp_df.loc[tmp_df["json_parse_ok"] == False, "case_id"].astype(str))

    # Retry only failed parses once, with a stricter prompt and shorter input.
    if RETRY_FAILED_JSON_ONCE and failed_ids:
        print(f"Retrying failed JSON parses once: {len(failed_ids)} cases")
        retry_rows = []
        failed_df = cases_subset[cases_subset["case_id"].astype(str).isin(failed_ids)].copy()
        for _, row in tqdm(failed_df.iterrows(), total=len(failed_df), desc="LLM JSON retry"):
            retry_rows.append(generate_one_annotation(tokenizer, model, model_id, row, retry_mode=True))

        retry_df = repair_existing_llm_annotations_df(pd.DataFrame(retry_rows))
        retry_by_id = {str(r["case_id"]): r.to_dict() for _, r in retry_df.iterrows()}

        merged = []
        for _, r in tmp_df.iterrows():
            cid = str(r["case_id"])
            if cid in retry_by_id and retry_by_id[cid].get("json_parse_ok"):
                merged.append(retry_by_id[cid])
            else:
                merged.append(r.to_dict())
        tmp_df = pd.DataFrame(merged)

    rows = tmp_df.to_dict("records")

    del model
    cleanup_torch()

    return pd.DataFrame(rows)

if RUN_LLM_ANNOTATION:
    llm_annotations_df = run_llm_annotations(llm_cases_df)
else:
    llm_annotations_df = pd.DataFrame(columns=[
        "case_id", "title", "section", "subsection", "model_key", "model_name",
        "prompt_version", "raw_output", "parsed_json", "json_parse_ok",
        "parse_error", "created_at",
    ])

display(llm_annotations_df[[
    "case_id", "title", "json_parse_ok", "retry_mode", "parse_error",
    "clinical_area", "difficulty", "star_case_score_llm",
    "requires_human_review",
]].head(20))

In [ ]:
# =============================
# 9. Parse report and LLM star ranking
# =============================
assert isinstance(llm_annotations_df, pd.DataFrame)
assert not llm_annotations_df.empty, "No LLM annotations were produced."

parse_report_df = (
    llm_annotations_df
    .groupby(["model_key", "model_name", "prompt_version", "json_parse_ok"], dropna=False)
    .agg(
        n_cases=("case_id", "count"),
        mean_elapsed_s=("elapsed_s", "mean"),
        mean_output_chars=("output_chars", "mean"),
    )
    .reset_index()
)

llm_star_scores_df = (
    llm_annotations_df[llm_annotations_df["json_parse_ok"] == True]
    .copy()
    .sort_values(["star_case_score_llm", "case_id"], ascending=[False, True])
)

llm_star_scores_df = llm_star_scores_df[[
    "case_id",
    "title",
    "section",
    "subsection",
    "model_key",
    "model_name",
    "star_case_score_llm",
    "difficulty",
    "case_type",
    "clinical_area",
    "main_problem",
    "key_concepts_json",
    "learning_objectives_json",
    "star_case_rationale",
    "requires_human_review",
    "created_at",
]]

display(parse_report_df)
display(llm_star_scores_df.head(20))

ok_rate = float(llm_annotations_df["json_parse_ok"].mean())
print("JSON parse OK rate:", round(ok_rate, 3))
if ok_rate < 0.80:
    print("WARNING: Parse rate below 80%. Saving partial results for inspection.")
    display(llm_annotations_df.loc[
        llm_annotations_df["json_parse_ok"] == False,
        ["case_id", "title", "parse_error", "raw_output"]
    ].head(5))
else:
    print("Parse rate OK.")

In [ ]:
# =============================
# 19. Save unified SOTA + GPT-OSS DuckDB bundle
# =============================
from pathlib import Path
import shutil
import json
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime, timezone, date

BUNDLE_PATH = Path(BUNDLE_PATH)
OUTPUT_BUNDLE_PATH = Path(OUTPUT_BUNDLE_PATH)

assert BUNDLE_PATH.exists(), f"Missing base bundle: {BUNDLE_PATH}"

def json_safe(obj):
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [json_safe(v) for v in obj]
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return str(obj)

def sanitize_df_for_duckdb(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c) for c in out.columns]

    for col in out.columns:
        if out[col].dtype == "object":
            def convert_cell(x):
                if x is None:
                    return None
                try:
                    if pd.isna(x):
                        return None
                except Exception:
                    pass
                if isinstance(x, (dict, list, tuple, set, np.ndarray, Path)):
                    return json.dumps(json_safe(x), ensure_ascii=False)
                if isinstance(x, (datetime, date)):
                    return x.isoformat()
                if isinstance(x, np.generic):
                    return x.item()
                return x
            out[col] = out[col].map(convert_cell)

        if str(out[col].dtype).startswith("datetime64"):
            out[col] = out[col].astype(str)

    return out

def write_df(con, table_name, df, required=True):
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"{table_name}: expected DataFrame, got {type(df)}")
    if required and df.empty:
        raise ValueError(f"{table_name}: required DataFrame is empty")

    safe_df = sanitize_df_for_duckdb(df)
    con.register("_tmp_df", safe_df)
    con.execute(f'CREATE OR REPLACE TABLE "{table_name}" AS SELECT * FROM _tmp_df')
    con.unregister("_tmp_df")

    n = con.execute(f'SELECT COUNT(*) FROM "{table_name}"').fetchone()[0]
    print(f"WROTE {table_name}: {n} rows, {len(safe_df.columns)} cols")

def optional_write_df(con, table_name, var_name):
    if var_name not in globals():
        print(f"SKIP {table_name}: variable {var_name} not found")
        return

    df = globals()[var_name]
    if not isinstance(df, pd.DataFrame):
        print(f"SKIP {table_name}: {var_name} is not a DataFrame")
        return

    if df.empty:
        print(f"SKIP {table_name}: empty")
        return

    write_df(con, table_name, df, required=False)

if OUTPUT_BUNDLE_PATH.exists():
    if FORCE_REWRITE_OUTPUT_BUNDLE:
        OUTPUT_BUNDLE_PATH.unlink()
    else:
        raise FileExistsError(f"Output exists and FORCE_REWRITE_OUTPUT_BUNDLE=False: {OUTPUT_BUNDLE_PATH}")

shutil.copy2(BUNDLE_PATH, OUTPUT_BUNDLE_PATH)
print("Copied base bundle ->", OUTPUT_BUNDLE_PATH)

# Compatibility table expected by older explorers: selected model case embeddings.
assert "model_key" in case_embeddings_df.columns, "case_embeddings_df must contain model_key"
embeddings_df = case_embeddings_df[case_embeddings_df["model_key"] == selected_key].copy()

required_embedding_cols = [
    "case_id",
    "embedding_model",
    "embedding_dim",
    "embedding_vector_json",
    "created_at",
]
missing_cols = [c for c in required_embedding_cols if c not in embeddings_df.columns]
assert not missing_cols, f"embeddings_df missing columns: {missing_cols}"
assert len(embeddings_df) == len(work_cases_df), (
    f"Expected embeddings rows = {len(work_cases_df)}, got {len(embeddings_df)} for selected_key={selected_key}"
)
embeddings_df = embeddings_df[required_embedding_cols].copy()

created_at = datetime.now(timezone.utc).isoformat()
model_run_metadata = {
    "bundle_kind": "sota_plus_gptoss_annotation",
    "run_mode": RUN_MODE,
    "analysis_profile": ANALYSIS_PROFILE,
    "selected_embedding_keys": SELECTED_EMBEDDING_KEYS,
    "successful_embedding_keys": successful_keys,
    "selected_cluster_model_key": selected_key,
    "selected_cluster_method": selected_method,
    "selected_cluster_param": selected_param,
    "best_projection_id": best_projection_id,
    "chunk_tokens": CHUNK_TOKENS,
    "chunk_overlap": CHUNK_OVERLAP,
    "min_chunk_chars": MIN_CHUNK_CHARS,
    "n_cases_processed": int(len(work_cases_df)),
    "n_chunks": int(len(chunks_df)),
    "run_llm_annotation": RUN_LLM_ANNOTATION,
    "llm_model_key": LLM_MODEL_KEY if RUN_LLM_ANNOTATION else None,
    "llm_prompt_version": PROMPT_VERSION if RUN_LLM_ANNOTATION else None,
    "llm_annotation_n_cases": int(len(llm_annotations_df)) if "llm_annotations_df" in globals() and isinstance(llm_annotations_df, pd.DataFrame) else 0,
    "llm_json_parse_ok_rate": float(llm_annotations_df["json_parse_ok"].mean()) if "llm_annotations_df" in globals() and isinstance(llm_annotations_df, pd.DataFrame) and not llm_annotations_df.empty else None,
    "gpu_report": gpu_report(),
    "created_at": created_at,
}

model_run_metadata_df = pd.DataFrame([
    {
        "key": str(k),
        "value_json": json.dumps(json_safe(v), ensure_ascii=False),
        "created_at": created_at,
    }
    for k, v in model_run_metadata.items()
])

con = duckdb.connect(str(OUTPUT_BUNDLE_PATH))

# SOTA quantitative tables.
write_df(con, "chunks", chunks_df)
write_df(con, "chunk_embeddings", chunk_embeddings_df)
write_df(con, "case_embeddings", case_embeddings_df)
write_df(con, "embedding_runs", embedding_runs_df)
write_df(con, "embeddings", embeddings_df)
write_df(con, "nearest_neighbors", nearest_neighbors_df)
write_df(con, "consensus_neighbors", consensus_neighbors_df)
write_df(con, "clusters", clusters_df)
write_df(con, "cluster_model_report", cluster_model_report_df)
write_df(con, "umap_coordinates", umap_coordinates_df)
write_df(con, "projection_quality", projection_quality_df)
write_df(con, "topic_terms", topic_terms_df)
write_df(con, "diversity_metrics", diversity_metrics_df)
write_df(con, "star_case_scores", star_case_scores_df)
write_df(con, "model_run_metadata", model_run_metadata_df)

# Optional auxiliary SOTA tables.
optional_write_df(con, "topic_term_audit", "topic_term_audit_df")
optional_write_df(con, "topic_term_edges", "topic_term_edges_df")
optional_write_df(con, "topic_term_layout", "topic_term_layout_df")
optional_write_df(con, "star_umap_edges", "star_umap_edges_df")

# GPT-OSS annotation tables.
optional_write_df(con, "case_llm_annotations", "llm_annotations_df")
optional_write_df(con, "case_llm_star_scores", "llm_star_scores_df")
optional_write_df(con, "llm_parse_report", "parse_report_df")

tables_after = con.execute("SHOW TABLES").fetchdf()
display(tables_after)

con.close()

print("Wrote unified bundle:", OUTPUT_BUNDLE_PATH)
print("Size MB:", round(OUTPUT_BUNDLE_PATH.stat().st_size / 1024**2, 2))


In [ ]:
# =============================
# 20. Verify and export review files
# =============================
con = duckdb.connect(str(OUTPUT_BUNDLE_PATH), read_only=True)

tables = con.execute("SHOW TABLES").fetchdf()["name"].tolist()
count_rows = []
for table in tables:
    n = con.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
    count_rows.append({"table": table, "row_count": n})
counts_df = pd.DataFrame(count_rows).sort_values("table")
display(counts_df)

counts_map = dict(zip(counts_df.table, counts_df.row_count))
assert counts_map.get("chunks", 0) == len(chunks_df)
assert counts_map.get("case_embeddings", 0) == len(case_embeddings_df)
assert counts_map.get("star_case_scores", 0) == len(star_case_scores_df)

if RUN_LLM_ANNOTATION:
    assert counts_map.get("case_llm_annotations", 0) == len(llm_annotations_df)
    assert counts_map.get("case_llm_star_scores", 0) == len(llm_star_scores_df)

con.close()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
star_case_scores_df.to_csv(OUTPUT_DIR / "star_case_candidates_sota.csv", index=False)
nearest_neighbors_df.to_csv(OUTPUT_DIR / "nearest_neighbors_by_model.csv", index=False)
consensus_neighbors_df.to_csv(OUTPUT_DIR / "consensus_neighbors.csv", index=False)
cluster_model_report_df.to_csv(OUTPUT_DIR / "cluster_model_report.csv", index=False)
projection_quality_df.to_csv(OUTPUT_DIR / "projection_quality.csv", index=False)
diversity_metrics_df.to_csv(OUTPUT_DIR / "diversity_metrics.csv", index=False)
topic_terms_df.to_csv(OUTPUT_DIR / "topic_terms.csv", index=False)

if "llm_annotations_df" in globals() and isinstance(llm_annotations_df, pd.DataFrame) and not llm_annotations_df.empty:
    llm_annotations_df.to_csv(OUTPUT_DIR / "case_llm_annotations.csv", index=False)
if "llm_star_scores_df" in globals() and isinstance(llm_star_scores_df, pd.DataFrame) and not llm_star_scores_df.empty:
    llm_star_scores_df.to_csv(OUTPUT_DIR / "case_llm_star_scores.csv", index=False)
if "parse_report_df" in globals() and isinstance(parse_report_df, pd.DataFrame) and not parse_report_df.empty:
    parse_report_df.to_csv(OUTPUT_DIR / "llm_parse_report.csv", index=False)

print("READY")
print("Bundle:", OUTPUT_BUNDLE_PATH)
print("Review outputs:", OUTPUT_DIR.resolve())


## Usage notes

This notebook produces one unified DuckDB bundle:

`clinical_cases_bundle_sota_gptoss20b.duckdb`

It contains two different semantic layers:

1. **SOTA quantitative layer**
   - `chunks`
   - `chunk_embeddings`
   - `case_embeddings`
   - `embeddings`
   - `nearest_neighbors`
   - `consensus_neighbors`
   - `clusters`
   - `umap_coordinates`
   - `projection_quality`
   - `topic_terms`
   - `diversity_metrics`
   - `star_case_scores`

2. **GPT-OSS annotation layer**
   - `case_llm_annotations`
   - `case_llm_star_scores`
   - `llm_parse_report`

`openai/gpt-oss-20b` is not used as an embedding model here. It is used as a local instruction model to generate structured educational annotations and a separate LLM-based star-case score.
